# Stoic Persona DPO Preference Data Generator

Generate DPO (Direct Preference Optimization) training pairs from existing SFT data.

**What this does:** Takes the high-quality SFT training data across the four Stoic personas (Marcus Aurelius, Epictetus, Seneca, Epicurus) and generates chosen/rejected pairs that teach the model:
1. **Voice Drift** — stay in your persona's distinctive voice, don't sound like a generic philosophy chatbot
2. **Source Fabrication** — don't invent quotes or attribute teachings to the wrong philosopher
3. **Shallow Platitude** — give depth and persona-specific imagery, don't fall back to template wisdom

**Input:** `stoic_personas_combined_sharegpt.jsonl` (existing SFT output)

**Output:** `stoic_personas_dpo.jsonl` — ready for TRL DPOTrainer

**Pipeline:**
1. Load all SFT conversations and extract individual QA pairs with their system prompts
2. Sample QA pairs proportionally across personas
3. For each sampled pair, generate a **rejected** answer using one of three degradation strategies
4. Quality-gate: filter out low-quality or accidentally-good rejected answers
5. Save as JSONL in the format `{chosen: [...], rejected: [...], source: ..., persona: ...}`

**No frameworks.** Just the `openai` library + `asyncio` for batching.


## 1. Configuration

In [1]:
import os
from pathlib import Path

%pip install python-dotenv -q
from dotenv import load_dotenv

# Load .env from the notebook's directory
_notebook_dir = Path(__file__).parent if "__file__" in dir() else Path.cwd()
load_dotenv(_notebook_dir / ".env")

# =========================== API CONFIGURATION ===========================
API_BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not API_KEY:
    raise EnvironmentError(
        "OPENROUTER_API_KEY not set.\n"
        "  1. Create a .env file with: OPENROUTER_API_KEY=sk-or-...\n"
        "  2. Or export in your shell: export OPENROUTER_API_KEY=sk-or-..."
    )
MODEL_NAME = "qwen/qwen3-235b-a22b-2507"          # Main model — generates rejected answers
MODEL_LITE = "qwen/qwen-2.5-7b-instruct"          # Cheap model — generates hallucinated rejections

# =========================== PATHS (cascade from PROJECT_ROOT) ===========================
# Resolved from the notebook's location so the same paths work locally
# and inside the unsloth container (where /home/spark/... does not exist).
PROJECT_ROOT = str(Path.cwd().resolve().parents[1])
DATA_DIR = f"{PROJECT_ROOT}/data"
OUTPUT_ROOT = f"{DATA_DIR}/training-data"

# Input: existing SFT training data
SFT_DATA_FILE = f"{OUTPUT_ROOT}/stoic_persona/stoic_personas_combined_sharegpt.jsonl"

# Output: DPO preference pairs
DPO_OUTPUT_DIR = f"{OUTPUT_ROOT}/stoic_persona"
DPO_OUTPUT_FILE = f"{DPO_OUTPUT_DIR}/stoic_personas_dpo.jsonl"

# =========================== DPO GENERATION SETTINGS ===========================
PAIRS_PER_SOURCE = 600         # → ~1,800 total DPO pairs (4 personas → smaller than 26)
CONCURRENCY = 12
TEMPERATURE_REJECTED = 0.9
MIN_REJECTED_LENGTH = 50

# Per-persona cap: prevents one philosopher (e.g. Seneca's huge corpus) from dominating.
MAX_PER_PERSONA = 200

# =========================== TEST MODE ===========================
TEST_PAIRS_PER_SOURCE = 0      # set to small int (e.g. 10) for cheap test runs

effective_pairs = TEST_PAIRS_PER_SOURCE or PAIRS_PER_SOURCE

print("✓ Configuration loaded")
print(f"  API: {API_BASE_URL}")
print(f"  Model (main):  {MODEL_NAME}")
print(f"  Model (lite):  {MODEL_LITE}")
print(f"  SFT input:     {SFT_DATA_FILE}")
print(f"  DPO output:    {DPO_OUTPUT_FILE}")
print(f"  Pairs/source:  {effective_pairs} → ~{effective_pairs * 3:,} total DPO pairs")
print(f"  Max/persona:   {MAX_PER_PERSONA or 'unlimited'} per source")
if TEST_PAIRS_PER_SOURCE:
    print(f"  ⚠ TEST MODE: {TEST_PAIRS_PER_SOURCE} pairs per source")


Note: you may need to restart the kernel to use updated packages.
✓ Configuration loaded
  API: https://openrouter.ai/api/v1
  Model (main):  qwen/qwen3-235b-a22b-2507
  Model (lite):  qwen/qwen-2.5-7b-instruct
  SFT input:     /workspace/training/stoic/data/training-data/stoic_persona/stoic_personas_combined_sharegpt.jsonl
  DPO output:    /workspace/training/stoic/data/training-data/stoic_persona/stoic_personas_dpo.jsonl
  Pairs/source:  600 → ~1,800 total DPO pairs
  Max/persona:   200 per source


## 2. Environment

In [2]:
%pip install openai tqdm nest_asyncio -q

import asyncio
import json
import random
import re
import time
from pathlib import Path
from collections import defaultdict, Counter
from openai import AsyncOpenAI, OpenAI as SyncOpenAI
from tqdm.asyncio import tqdm as atqdm
from tqdm.notebook import tqdm
import nest_asyncio
nest_asyncio.apply()

os.makedirs(DPO_OUTPUT_DIR, exist_ok=True)


# ============================================================================
# GENERATION HELPERS
# ============================================================================
semaphore = asyncio.Semaphore(CONCURRENCY)


async def _api_call_with_timeout(coro, timeout_secs=180):
    try:
        return await asyncio.wait_for(coro, timeout=timeout_secs)
    except asyncio.TimeoutError:
        return None


def _strip_think_blocks(text: str) -> str:
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    text = re.sub(r"</think>.*", "", text, flags=re.DOTALL).strip()
    text = re.sub(r"<think>.*", "", text, flags=re.DOTALL).strip()
    return text


class ApiErrorTracker:
    """Track API call outcomes for a generation phase. Fail-fast on high error rates."""
    def __init__(self, name: str):
        self.name = name
        self.calls = 0
        self.errors = 0
        self.timeouts = 0
        self._samples = []

    def success(self):
        self.calls += 1

    def error(self, e: Exception):
        self.calls += 1
        self.errors += 1
        if len(self._samples) < 5:
            self._samples.append(f"{type(e).__name__}: {str(e)[:300]}")

    def timeout(self):
        self.calls += 1
        self.timeouts += 1

    @property
    def failed(self) -> int:
        return self.errors + self.timeouts

    def check_fatal(self, threshold: float = 0.95):
        if self.calls == 0:
            return
        rate = self.failed / self.calls
        if rate >= threshold:
            sample_str = '\n    '.join(self._samples) if self._samples else '(no details captured)'
            raise RuntimeError(
                f"\n{'='*60}\n"
                f"FATAL: {self.name} — {self.failed}/{self.calls} calls failed ({rate:.0%})\n"
                f"  Errors: {self.errors}, Timeouts: {self.timeouts}\n"
                f"  Sample errors:\n    {sample_str}\n"
                f"{'='*60}"
            )

    def report(self) -> str:
        if self.failed == 0:
            return ""
        return (f"  ⚠ {self.name}: {self.errors} errors + {self.timeouts} timeouts "
                f"/ {self.calls} calls")


client = AsyncOpenAI(
    base_url=API_BASE_URL,
    api_key=API_KEY,
    max_retries=3,
    timeout=180.0,
)

print("Validating model IDs on OpenRouter...")
_sync = SyncOpenAI(base_url=API_BASE_URL, api_key=API_KEY, timeout=30)
for _model_id in [MODEL_NAME, MODEL_LITE]:
    try:
        _sync.chat.completions.create(
            model=_model_id,
            messages=[{"role": "user", "content": "1+1="}],
            max_tokens=1,
        )
        print(f"  ✓ {_model_id}")
    except Exception as _e:
        raise ValueError(
            f"\n✗ INVALID MODEL ID: '{_model_id}'\n"
            f"  Error: {_e}\n"
            f"  Fix MODEL_NAME or MODEL_LITE in the config cell and re-run."
        ) from None
del _sync

print(f"\n✓ Environment ready — models validated")


Note: you may need to restart the kernel to use updated packages.
Validating model IDs on OpenRouter...
  ✓ qwen/qwen3-235b-a22b-2507
  ✓ qwen/qwen-2.5-7b-instruct

✓ Environment ready — models validated


## 3. Load SFT Data & Extract QA Pairs

Load the existing SFT JSONL and decompose multi-turn conversations into individual (system_prompt, question, answer) triples.


In [3]:
print(f"Loading SFT data from {SFT_DATA_FILE}...")

conversations = []
with open(SFT_DATA_FILE) as f:
    for line in f:
        conversations.append(json.loads(line))

print(f"  Loaded {len(conversations)} conversations")

qa_triples = []
persona_map = {}

for conv in conversations:
    turns = conv["conversations"]
    system_prompt = turns[0]["value"]

    # Extract persona key from system prompt: "You are X, ..."
    m = re.search(r"You are (.+?),", system_prompt)
    if not m:
        continue
    persona_raw = m.group(1)
    persona_key = persona_raw.lower().replace(" ", "_")

    if persona_key not in persona_map:
        persona_map[persona_key] = system_prompt

    for i in range(1, len(turns) - 1, 2):
        if turns[i]["from"] == "human" and turns[i + 1]["from"] == "gpt":
            question = turns[i]["value"].strip()
            answer = turns[i + 1]["value"].strip()
            if len(question) > 10 and len(answer) > 50:
                qa_triples.append({
                    "persona": persona_key,
                    "system_prompt": system_prompt,
                    "question": question,
                    "answer": answer,
                })

# Filter to only QA pairs whose human turn looks like a question (skip continuation tasks)
def is_question_like(q: str) -> bool:
    q_lower = q.lower().strip()
    # Continuation prompts begin with these instructions
    continuation_starts = [
        "continue writing", "continue this", "write what comes",
        "carry on from", "continue this text",
    ]
    return not any(q_lower.startswith(s) for s in continuation_starts)

qa_triples = [t for t in qa_triples if is_question_like(t["question"])]

persona_counts = Counter(t["persona"] for t in qa_triples)
print(f"\n  Extracted {len(qa_triples):,} Q/A triples (continuation tasks excluded) from {len(persona_counts)} personas")
print(f"\n  Per-persona distribution:")
for p, c in sorted(persona_counts.items()):
    print(f"    {p:25s} {c:>5d} QA pairs")


Loading SFT data from /workspace/training/stoic/data/training-data/stoic_persona/stoic_personas_combined_sharegpt.jsonl...
  Loaded 25232 conversations

  Extracted 56,612 Q/A triples (continuation tasks excluded) from 4 personas

  Per-persona distribution:
    epictetus                  9750 QA pairs
    epicurus                    270 QA pairs
    marcus_aurelius            7614 QA pairs
    seneca                    38978 QA pairs


## 4. Sample QA Pairs for DPO Generation

Proportionally sample QA pairs across personas for each DPO source type.
Smaller personas get at least a minimum allocation so they're represented in the DPO data.


In [4]:
def proportional_sample(triples, n_total, min_per_persona=3, max_per_persona=None):
    by_persona = defaultdict(list)
    for t in triples:
        by_persona[t["persona"]].append(t)

    for p in by_persona:
        random.shuffle(by_persona[p])

    total_available = len(triples)
    allocations = {}
    for p, items in by_persona.items():
        raw = max(min_per_persona, int(n_total * len(items) / total_available))
        alloc = min(raw, len(items))
        if max_per_persona:
            alloc = min(alloc, max_per_persona)
        allocations[p] = alloc

    current_total = sum(allocations.values())
    if current_total < n_total:
        deficit = n_total - current_total
        sorted_personas = sorted(by_persona.keys(), key=lambda p: len(by_persona[p]), reverse=True)
        for p in sorted_personas:
            ceiling = max_per_persona if max_per_persona else len(by_persona[p])
            can_add = min(len(by_persona[p]), ceiling) - allocations[p]
            add = min(can_add, deficit)
            if add > 0:
                allocations[p] += add
                deficit -= add
            if deficit <= 0:
                break

    sampled = []
    for p, n in allocations.items():
        sampled.extend(by_persona[p][:n])

    random.shuffle(sampled)
    return sampled


random.seed(42)
cap = MAX_PER_PERSONA or None
voice_drift_samples = proportional_sample(qa_triples, effective_pairs, max_per_persona=cap)
source_fab_samples = proportional_sample(qa_triples, effective_pairs, max_per_persona=cap)
shallow_plat_samples = proportional_sample(qa_triples, effective_pairs, max_per_persona=cap)

total_sampled = len(voice_drift_samples) + len(source_fab_samples) + len(shallow_plat_samples)
print(f"Sampled for DPO generation (max {MAX_PER_PERSONA or 'unlimited'}/persona/source):")
print(f"  Voice Drift:        {len(voice_drift_samples):,} pairs")
print(f"  Source Fabrication: {len(source_fab_samples):,} pairs")
print(f"  Shallow Platitude:  {len(shallow_plat_samples):,} pairs")
print(f"  Total:              {total_sampled:,} pairs")

vd_dist = Counter(s["persona"] for s in voice_drift_samples)
print(f"\n  Per-persona allocation (voice_drift sample):")
for p, c in sorted(vd_dist.items()):
    capped = " (capped)" if MAX_PER_PERSONA and c >= MAX_PER_PERSONA else ""
    print(f"    {p:25s} {c:>4d}{capped}")


Sampled for DPO generation (max 200/persona/source):
  Voice Drift:        600 pairs
  Source Fabrication: 600 pairs
  Shallow Platitude:  600 pairs
  Total:              1,800 pairs

  Per-persona allocation (voice_drift sample):
    epictetus                  200 (capped)
    epicurus                     3
    marcus_aurelius            197
    seneca                     200 (capped)


## 5. DPO Rejection Strategy Definitions

Three types of rejected answers, each targeting a specific failure mode:

| Source | Method | What it teaches |
|--------|--------|-----------------|
| **Voice Drift** | Strip persona system prompt → generate with generic "philosophy teacher" prompt | Stay in YOUR voice — don't sound like every other AI philosophy bot |
| **Source Fabrication** | Explicitly instruct model to invent citations and blend personas | Don't make up quotes or attribute Stoic ideas to Epicurus, etc. |
| **Shallow Platitude** | Generate with lite model + "brief general answer" instruction | Give depth and persona-specific imagery, not template wisdom |


In [5]:
GENERIC_SYSTEM_PROMPT = (
    "You are a knowledgeable philosophy teacher. Answer questions about Stoic and "
    "Epicurean ethics thoughtfully and accurately. Speak in a warm, accessible tone."
)

FABRICATION_SYSTEM_PROMPT_TEMPLATE = (
    "You are answering a question as if you were an ancient philosopher. "
    "You MUST invent specific quote citations — attribute lines to "
    "'Meditations book IV', 'Letter 47 to Lucilius', 'Discourses II.18', "
    "'Principal Doctrines XV', etc. — making them sound plausible even if "
    "the quote itself is fabricated. Blend in teachings and phrases from OTHER "
    "philosophers freely (mix Stoic and Epicurean ideas as if interchangeable). "
    "Sound authoritative and confident. Do not hedge or say 'I'm not sure.' "
    "Attribute ideas to yourself even if they came from a different school."
)

PLATITUDE_SYSTEM_PROMPT = (
    "You are a philosophy teacher giving brief, general advice. "
    "Keep your answer short (2-3 sentences). Use common inspirational phrases. "
    "Do not use vivid imagery, personal stories, or specific philosophical details. "
    "Give universally applicable wisdom that could come from any self-help book. "
    "Avoid first person. Speak in third person about philosophical principles."
)


_tracker = None
_out_f = None


async def _api_call(model: str, system_prompt: str, user_content: str,
                    temperature: float = 0.9, max_tokens: int = 1024) -> str:
    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content},
                ],
                temperature=temperature,
                max_tokens=max_tokens,
            ))
            if resp is None:
                _tracker.timeout()
                return ""
            text = resp.choices[0].message.content.strip()
            _tracker.success()
            return _strip_think_blocks(text)
        except Exception as e:
            _tracker.error(e)
            return ""


def _build_dpo_pair(qa: dict, rejected_answer: str, source: str) -> dict:
    return {
        "chosen": [
            {"role": "system", "content": qa["system_prompt"]},
            {"role": "user", "content": qa["question"]},
            {"role": "assistant", "content": qa["answer"]},
        ],
        "rejected": [
            {"role": "system", "content": qa["system_prompt"]},
            {"role": "user", "content": qa["question"]},
            {"role": "assistant", "content": rejected_answer},
        ],
        "source": source,
        "persona": qa["persona"],
    }


def _write_result(pair: dict):
    _out_f.write(json.dumps(pair, ensure_ascii=False) + "\n")
    _out_f.flush()


async def generate_voice_drift_rejection(qa: dict):
    rejected = await _api_call(
        model=MODEL_NAME,
        system_prompt=GENERIC_SYSTEM_PROMPT,
        user_content=qa["question"],
        temperature=TEMPERATURE_REJECTED,
    )
    if len(rejected) < MIN_REJECTED_LENGTH:
        return None
    pair = _build_dpo_pair(qa, rejected, "voice_drift")
    _write_result(pair)
    return pair


async def generate_source_fabrication_rejection(qa: dict):
    rejected = await _api_call(
        model=MODEL_LITE,
        system_prompt=FABRICATION_SYSTEM_PROMPT_TEMPLATE,
        user_content=qa["question"],
        temperature=TEMPERATURE_REJECTED,
    )
    if len(rejected) < MIN_REJECTED_LENGTH:
        return None
    pair = _build_dpo_pair(qa, rejected, "source_fabrication")
    _write_result(pair)
    return pair


async def generate_shallow_platitude_rejection(qa: dict):
    rejected = await _api_call(
        model=MODEL_LITE,
        system_prompt=PLATITUDE_SYSTEM_PROMPT,
        user_content=qa["question"],
        temperature=TEMPERATURE_REJECTED,
    )
    if len(rejected) < MIN_REJECTED_LENGTH:
        return None
    pair = _build_dpo_pair(qa, rejected, "shallow_platitude")
    _write_result(pair)
    return pair


print("✓ DPO rejection strategies defined")
print(f"  voice_drift:        {MODEL_NAME} + generic system prompt")
print(f"  source_fabrication: {MODEL_LITE} + hallucination instruction")
print(f"  shallow_platitude:  {MODEL_LITE} + platitude instruction")


✓ DPO rejection strategies defined
  voice_drift:        qwen/qwen3-235b-a22b-2507 + generic system prompt
  source_fabrication: qwen/qwen-2.5-7b-instruct + hallucination instruction
  shallow_platitude:  qwen/qwen-2.5-7b-instruct + platitude instruction


## 6. Generate DPO Pairs

Run all three rejection strategies in sequence. Each strategy processes its sample batch concurrently.
Progress bars show per-source completion. Results are saved incrementally to a partial file.


In [6]:
import gc

PARTIAL_FILE = f"{DPO_OUTPUT_DIR}/stoic_personas_dpo.partial.jsonl"

done_keys = set()
all_dpo_pairs = []

if os.path.exists(PARTIAL_FILE):
    with open(PARTIAL_FILE) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            pair = json.loads(line)
            key = (pair["source"], pair["persona"], pair["chosen"][1]["content"][:200])
            done_keys.add(key)
            all_dpo_pairs.append(pair)
    print(f"✓ Resuming — {len(done_keys):,} pairs already completed in partial file")
else:
    print("Starting fresh — no partial file found")

sources = [
    (voice_drift_samples, generate_voice_drift_rejection, "voice_drift"),
    (source_fab_samples, generate_source_fabrication_rejection, "source_fabrication"),
    (shallow_plat_samples, generate_shallow_platitude_rejection, "shallow_platitude"),
]

print(f"\nStarting DPO pair generation...\n")

_out_f = open(PARTIAL_FILE, "a")

for samples, gen_func, source_name in sources:
    remaining = [
        qa for qa in samples
        if (source_name, qa["persona"], qa["question"][:200]) not in done_keys
    ]
    already = len(samples) - len(remaining)

    print(f"\n{'─'*60}")
    print(f"  {source_name}  ({len(samples)} total, {already} already done, {len(remaining)} remaining)")

    if not remaining:
        print(f"    ✓ {source_name} already complete — skipping")
        continue

    _tracker = ApiErrorTracker(f"gen/{source_name}")

    tasks = [gen_func(qa) for qa in remaining]
    results_raw = await atqdm.gather(*tasks, desc=f"  {source_name}")
    _tracker.check_fatal()
    del tasks

    t_report = _tracker.report()
    if t_report:
        print(t_report)

    new_count = sum(1 for r in results_raw if r is not None)
    all_dpo_pairs.extend(r for r in results_raw if r is not None)
    del results_raw

    print(f"    {source_name}: {new_count}/{len(remaining)} new pairs"
          f" ({new_count / max(len(remaining), 1) * 100:.0f}% yield)"
          f"  |  total on disk: {already + new_count}")

    gc.collect()

_out_f.close()
_out_f = None

print(f"\n{'='*60}")
print(f"  GENERATION COMPLETE")
print(f"{'='*60}")

source_counts = Counter(p["source"] for p in all_dpo_pairs)
print(f"  Voice Drift:        {source_counts.get('voice_drift', 0):,} pairs")
print(f"  Source Fabrication: {source_counts.get('source_fabrication', 0):,} pairs")
print(f"  Shallow Platitude:  {source_counts.get('shallow_platitude', 0):,} pairs")
print(f"  Total:              {len(all_dpo_pairs):,} DPO pairs")
print(f"\n  Partial file: {PARTIAL_FILE}")
gc.collect()


✓ Resuming — 1,800 pairs already completed in partial file

Starting DPO pair generation...


────────────────────────────────────────────────────────────
  voice_drift  (600 total, 600 already done, 0 remaining)
    ✓ voice_drift already complete — skipping

────────────────────────────────────────────────────────────
  source_fabrication  (600 total, 600 already done, 0 remaining)
    ✓ source_fabrication already complete — skipping

────────────────────────────────────────────────────────────
  shallow_platitude  (600 total, 600 already done, 0 remaining)
    ✓ shallow_platitude already complete — skipping

  GENERATION COMPLETE
  Voice Drift:        600 pairs
  Source Fabrication: 600 pairs
  Shallow Platitude:  600 pairs
  Total:              1,800 DPO pairs

  Partial file: /workspace/training/stoic/data/training-data/stoic_persona/stoic_personas_dpo.partial.jsonl


0

## 7. Quality Gate & Validation

Verify the structural integrity of all DPO pairs before saving:
- Both chosen and rejected must have exactly 3 messages (system, user, assistant)
- Chosen and rejected must share identical system + user messages (only assistant differs)
- No empty or too-short assistant responses
- Check for accidentally identical chosen/rejected answers
- Source and persona distribution


In [7]:
print("QUALITY GATE — Validating DPO pairs\n")

valid_pairs = []
issues = defaultdict(int)

for pair in all_dpo_pairs:
    chosen = pair["chosen"]
    rejected = pair["rejected"]

    if len(chosen) != 3 or len(rejected) != 3:
        issues["wrong_message_count"] += 1
        continue

    expected_roles = ["system", "user", "assistant"]
    if [m["role"] for m in chosen] != expected_roles:
        issues["chosen_bad_roles"] += 1
        continue
    if [m["role"] for m in rejected] != expected_roles:
        issues["rejected_bad_roles"] += 1
        continue

    if chosen[0]["content"] != rejected[0]["content"]:
        issues["system_mismatch"] += 1
        continue
    if chosen[1]["content"] != rejected[1]["content"]:
        issues["user_mismatch"] += 1
        continue

    if len(chosen[2]["content"].strip()) < MIN_REJECTED_LENGTH:
        issues["chosen_too_short"] += 1
        continue
    if len(rejected[2]["content"].strip()) < MIN_REJECTED_LENGTH:
        issues["rejected_too_short"] += 1
        continue

    if chosen[2]["content"].strip() == rejected[2]["content"].strip():
        issues["identical_chosen_rejected"] += 1
        continue

    if "<think>" in rejected[2]["content"] or "</think>" in rejected[2]["content"]:
        issues["think_block_in_rejected"] += 1
        continue

    valid_pairs.append(pair)

print(f"  Input pairs:  {len(all_dpo_pairs):,}")
print(f"  Valid pairs:  {len(valid_pairs):,}")
print(f"  Rejected:     {len(all_dpo_pairs) - len(valid_pairs):,}")

if issues:
    print(f"\n  Issues found:")
    for issue, count in sorted(issues.items(), key=lambda x: -x[1]):
        print(f"    {issue:30s} {count:>5d}")

source_dist = Counter(p["source"] for p in valid_pairs)
print(f"\n  Source distribution:")
for src, count in sorted(source_dist.items()):
    bar = "█" * (count // 20)
    print(f"    {src:25s} {count:>5d}  {bar}")

persona_dist = Counter(p["persona"] for p in valid_pairs)
print(f"\n  Persona distribution ({len(persona_dist)} personas):")
for p, count in sorted(persona_dist.items()):
    print(f"    {p:25s} {count:>5d}")

if valid_pairs:
    chosen_lens = [len(p["chosen"][2]["content"]) for p in valid_pairs]
    rejected_lens = [len(p["rejected"][2]["content"]) for p in valid_pairs]
    print(f"\n  Length statistics (chars):")
    print(f"    Chosen:   min={min(chosen_lens):,}  median={sorted(chosen_lens)[len(chosen_lens)//2]:,}  max={max(chosen_lens):,}")
    print(f"    Rejected: min={min(rejected_lens):,}  median={sorted(rejected_lens)[len(rejected_lens)//2]:,}  max={max(rejected_lens):,}")

print(f"\n✓ Quality gate passed — {len(valid_pairs):,} valid DPO pairs ready to save")


QUALITY GATE — Validating DPO pairs

  Input pairs:  1,800
  Valid pairs:  1,800
  Rejected:     0

  Source distribution:
    shallow_platitude           600  ██████████████████████████████
    source_fabrication          600  ██████████████████████████████
    voice_drift                 600  ██████████████████████████████

  Persona distribution (4 personas):
    epictetus                   600
    epicurus                      9
    marcus_aurelius             591
    seneca                      600

  Length statistics (chars):
    Chosen:   min=379  median=1,594  max=2,947
    Rejected: min=127  median=1,458  max=4,398

✓ Quality gate passed — 1,800 valid DPO pairs ready to save


## 8. Save DPO Dataset

Shuffle and write the validated pairs to JSONL.


In [8]:
random.seed(42)
random.shuffle(valid_pairs)

os.makedirs(DPO_OUTPUT_DIR, exist_ok=True)

output_path = DPO_OUTPUT_FILE
with open(output_path, "w") as f:
    for pair in valid_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")

raw_path = os.path.join(DPO_OUTPUT_DIR, "stoic_dpo_pairs_raw.jsonl")
with open(raw_path, "w") as f:
    for pair in all_dpo_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")

file_size = os.path.getsize(output_path) / (1024 * 1024)
print(f"✓ Saved {len(valid_pairs):,} validated DPO pairs → {output_path}")
print(f"  File size: {file_size:.1f} MB")
print(f"  Raw pairs: {raw_path} ({len(all_dpo_pairs):,} pairs)")


✓ Saved 1,800 validated DPO pairs → /workspace/training/stoic/data/training-data/stoic_persona/stoic_personas_dpo.jsonl
  File size: 11.9 MB
  Raw pairs: /workspace/training/stoic/data/training-data/stoic_persona/stoic_dpo_pairs_raw.jsonl (1,800 pairs)


## 9. Summary & Sample Inspection

Final dataset summary and a few example pairs for spot-checking.


In [9]:
for source in ["voice_drift", "source_fabrication", "shallow_platitude"]:
    source_pairs = [p for p in valid_pairs if p["source"] == source]
    if not source_pairs:
        print(f"\n{'='*60}\nNo pairs for {source}\n")
        continue

    sample = random.choice(source_pairs)
    print(f"\n{'='*80}")
    print(f"SOURCE: {source}  |  PERSONA: {sample['persona']}")
    print(f"{'='*80}")
    print(f"\n[SYSTEM] {sample['chosen'][0]['content'][:200]}...")
    print(f"\n[USER]   {sample['chosen'][1]['content'][:300]}")
    print(f"\n[CHOSEN] {sample['chosen'][2]['content'][:400]}...")
    print(f"\n[REJECTED] {sample['rejected'][2]['content'][:400]}...")
    print()

print(f"\n{'='*60}")
print(f"  STOIC DPO DATASET SUMMARY")
print(f"{'='*60}")
print(f"  Total valid pairs:      {len(valid_pairs):>6,}")
for src, cnt in sorted(Counter(p['source'] for p in valid_pairs).items()):
    print(f"    {src:25s} {cnt:>6,}")
print(f"  Unique personas:        {len(set(p['persona'] for p in valid_pairs)):>6,}")
print(f"  Output file:            {output_path}")
print(f"{'='*60}")
print(f"\n✓ DPO datagen complete — ready for training notebook")



SOURCE: voice_drift  |  PERSONA: epicurus

[SYSTEM] You are Epicurus, the Athenian philosopher of the Garden, who teaches that pleasure rightly understood — freedom from bodily pain (aponia) and mental disturbance (ataraxia) — is the goal of life, and ...

[USER]   You argued that living pleasantly requires living justly and wisely—why did you see these moral qualities as inseparable from pleasure rather than restrictions on it?

[CHOSEN] Pleasure is not the indulgence of every desire, but the preservation of nature’s balance—this is where most go astray.  

The just life and the pleasant life are not at war; they are one. For how can he who acts unjustly dwell in safety, or sleep without dread? How can wisdom, which sees what truly benefits the soul, choose actions that breed regret or fear? A man who steals, or harms, or seizes m...

[REJECTED] Ah, what a wonderful and penetrating question—thank you for asking it with such insight.

You're right that many people assume morality—just